# Kaggle Digit Recognizer
## 3-Block CNN Architecture (PyTorch)

Based on the community's research, this notebook implements a legitimate >99.4% accuracy model without data leakage. It features:
- A deep 3-block Convolutional Neural Network
- Heavy use of `BatchNorm` and `Dropout` (0.25 for conv, 0.5 for dense)
- Careful Data Augmentation (Rotation 10°, Shift 10%, strictly no flips)
- `ReduceLROnPlateau` for learning rate scheduling

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Load data
# It checks local directory first, then Kaggle's typical input directory
train_path = '../train.csv' if os.path.exists('../train.csv') else '/kaggle/input/digit-recognizer/train.csv'
test_path = '../test.csv' if os.path.exists('../test.csv') else '/kaggle/input/digit-recognizer/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

In [ ]:
# Prepare datasets
class MNISTDataset(Dataset):
    def __init__(self, df, transform=None, is_test=False):
        self.is_test = is_test
        self.transform = transform
        
        if not is_test:
            self.labels = df['label'].values
            self.images = df.drop('label', axis=1).values.astype(np.float32)
        else:
            self.images = df.values.astype(np.float32)
            
        # Reshape to 28x28 and normalize to [0, 1]
        self.images = self.images.reshape(-1, 28, 28) / 255.0
        
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        # Convert to shape (1, 28, 28) for PyTorch
        img = self.images[idx]
        img_tensor = torch.tensor(img).unsqueeze(0)
        
        if self.transform:
            img_tensor = self.transform(img_tensor)
            
        if not self.is_test:
            label = torch.tensor(self.labels[idx], dtype=torch.long)
            return img_tensor, label
        return img_tensor

# Define Transforms (Careful Data Augmentation based on community findings)
train_transforms = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    # Strictly no flips as they destroy digit orientations (e.g. 6 -> 9)
])

# Split into train and validation (90-10 split)
train_data, val_data = train_test_split(train_df, test_size=0.1, random_state=42)

train_dataset = MNISTDataset(train_data, transform=train_transforms)
val_dataset = MNISTDataset(val_data, transform=None)
test_dataset = MNISTDataset(test_df, transform=None, is_test=True)

train_loader = DataLoader(train_dataset, batch_size=86, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=86, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=86, shuffle=False)

In [ ]:
# Define 3-Block CNN Architecture
class DeepCNN(nn.Module):
    def __init__(self):
        super(DeepCNN, self).__init__()
        
        # Block 1
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )
        
        # Block 2
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )
        
        # Block 3
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, padding=1), # padding=1 handles odd dimensions before pooling
            nn.Dropout2d(0.25)
        )
        
        # Dense/Classification Head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 10)
        )
        
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.classifier(x)
        return x

model = DeepCNN().to(device)
print("Model Parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
# Training Setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, verbose=True)

In [ ]:
# Training Loop
epochs = 30
best_val_acc = 0.0

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    # Validation phase
    model.eval()
    correct = 0
    total = 0
    val_loss = 0.0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    val_acc = 100 * correct / total
    avg_train_loss = running_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    
    print(f'Epoch [{epoch+1}/{epochs}] - Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f} - Val Acc: {val_acc:.2f}%')
    
    # LR Scheduler Step
    scheduler.step(val_acc)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')

print(f"\nBest Validation Accuracy: {best_val_acc:.2f}%")

In [ ]:
# Load best model for prediction
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

predictions = []

with torch.no_grad():
    for images in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        predictions.extend(predicted.cpu().numpy())

# Create submission file
submission = pd.DataFrame({
    'ImageId': range(1, len(predictions) + 1),
    'Label': predictions
})
submission.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")
submission.head()